# Prioritizing candidate genes using the moment of inertia tensor

Reference implementation for:

> Thummadi NB, Mallikarjuna T, Vindal V, Manimaran P.
> **Prioritizing the candidate genes related to cervical cancer using the moment of inertia tensor.**
> *Proteins.* 2022;90(2):363-371. doi:[10.1002/prot.26226](https://doi.org/10.1002/prot.26226)

## Idea in brief
Each amino acid is mapped to a fixed point in 3D space (a point on a circle in the
x-y plane, with a +/-1 z coordinate encoding a physicochemical property). A protein
sequence therefore becomes a cloud of points. For that cloud we compute the
**moment of inertia tensor** and take its three eigenvalues as a compact,
alignment-free descriptor of the sequence.

The pipeline then:
1. Encodes every experimentally verified and every candidate protein into its
   3-eigenvalue descriptor.
2. Builds a Euclidean **distance matrix** between candidate and known genes.
3. Thresholds the matrix into a binary similarity matrix and scores each candidate
   by how many known genes it is similar to.
4. Sorts / prioritizes candidates and draws a **dendrogram**.

## Configuration
Set your NCBI Entrez email (required by NCBI for E-utilities access) and the path
to the input CSV. Paths are relative to the repository root so the notebook runs
anywhere after `git clone`.

In [ ]:
# --- User configuration -----------------------------------------------------
# NCBI requires a contact email for Entrez / E-utilities requests.
# Set your own address here (or export it as the ENTREZ_EMAIL env variable).
import os
ENTREZ_EMAIL = os.environ.get("ENTREZ_EMAIL", "your.email@example.com")

# Input file: two pairs of columns -> (Candidate gene, UniProt Entry) and
# (Experimental gene, UniProt Entry). See data/fretrieve.csv.
DATA_PATH = os.path.join("..", "data", "fretrieve.csv")
# ---------------------------------------------------------------------------

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import eig

from Bio import Entrez, SeqIO
Entrez.email = ENTREZ_EMAIL

## 1. Amino-acid coordinate map and the moment-of-inertia descriptor
Each of the 20 amino acids (plus ambiguity codes) is placed at a fixed
`(x, y, z)` location. `eig_val` centers the point cloud on its centre of mass,
builds the 3x3 inertia tensor, and returns its eigenvalues.

In [ ]:
# Fixed 3D coordinates for each residue (unit circle in x-y, z encodes a property).
RESIDUE_COORDS = {
    'A': (0.9511,  0.3090, -1), 'B': (0.9511,  0.3090, -1),
    'C': (0.8090,  0.5878, -1), 'D': (-0.9511, -0.3090,  1),
    'E': (0.3090, -0.9511,  1), 'F': (-0.3090,  0.9511,  1),
    'G': (0.5878, -0.8090, -1), 'H': (0.8090, -0.5878,  1),
    'I': (-0.5878, 0.8090, -1), 'J': (-0.5878, 0.8090, -1),
    'K': (-0.8090, -0.5878, 1), 'L': (-0.8090, 0.5878, -1),
    'M': (0.5878,  0.8090,  1), 'N': (-0.5878, -0.8090, -1),
    'O': (-0.5878, -0.8090, -1),'P': (0.3090,  0.9511, -1),
    'Q': (0.9511, -0.3090,  1), 'R': (-0.3090, -0.9511, 1),
    'S': (0.0,    -1.0,     -1),'T': (1.0,     0.0,     -1),
    'U': (1.0,     0.0,     -1),'V': (0.0,     1.0,     -1),
    'W': (-0.9511, 0.3090,  1), 'X': (-0.9511, 0.3090,  1),
    'Y': (-1.0,    0.0,      1), 'Z': (-1.0,    0.0,     1),
}


def eig_val(sequence):
    """Return the 3 eigenvalues of the moment-of-inertia tensor of a sequence."""
    xs, ys, zs = [], [], []
    for residue in sequence:
        x, y, z = RESIDUE_COORDS[residue]
        xs.append(x); ys.append(y); zs.append(z)

    # Shift to the centre of mass.
    xs = np.asarray(xs) - np.mean(xs)
    ys = np.asarray(ys) - np.mean(ys)
    zs = np.asarray(zs) - np.mean(zs)

    ixx = np.sum(ys**2 + zs**2)
    iyy = np.sum(xs**2 + zs**2)
    izz = np.sum(xs**2 + ys**2)
    ixy = np.sum(xs * ys)
    iyz = np.sum(ys * zs)
    ixz = np.sum(xs * zs)

    inertia = np.array([[ ixx, -ixy, -ixz],
                        [-ixy,  iyy, -iyz],
                        [-ixz, -iyz,  izz]])
    eigenvalues, _ = eig(inertia)
    return eigenvalues

## 2. Fetch sequences from NCBI and encode them
`retrieve` takes a list of accession IDs, downloads the FASTA sequences from the
NCBI protein database, and returns the eigenvalue descriptor for each.

In [ ]:
def retrieve(accession_ids):
    """Fetch FASTA records for the given IDs and return their eigenvalue descriptors."""
    handle = Entrez.efetch(db="protein", id=accession_ids,
                           rettype="fasta", retmode="text")
    records = list(SeqIO.parse(handle, "fasta"))
    handle.close()
    return [eig_val(rec.seq) for rec in records]

## 3. Load candidate and experimental gene lists
`fretrieve.csv` holds two pairs of columns. Column index 1 is the UniProt entry
for the **candidate** genes; the last column is the UniProt entry for the
**experimentally verified** genes.

In [ ]:
genes = pd.read_csv(DATA_PATH)

# Predicted / candidate entries (column index 1) and experimental entries (last column).
predicted    = list(np.array(genes.iloc[:, 1]).flatten())
experimental = list(np.array(genes.iloc[:, -1]).flatten())

# Drop trailing NaNs (the two lists have different lengths).
predicted    = [g for g in predicted    if isinstance(g, str)]
experimental = [g for g in experimental if isinstance(g, str)]

print(f"{len(predicted)} candidate genes, {len(experimental)} experimental genes")

## 4. Build the distance matrix
Euclidean distance between the eigenvalue descriptors of every
experimental gene (rows) and every candidate gene (columns).

> **Note:** downloading thousands of sequences from NCBI can take a long time and
> is rate-limited. For large lists, batch the requests or cache the FASTA files.

In [ ]:
def distance_matrix(row_ids, col_ids):
    """Euclidean distance matrix between two sets of accession IDs."""
    rows = retrieve(row_ids)
    cols = retrieve(col_ids)
    dist = np.array([[np.linalg.norm(r - c) for c in cols] for r in rows])
    df = pd.DataFrame(dist, index=row_ids, columns=col_ids)
    return df


# Example (uncomment to run against NCBI):
# dmat = distance_matrix(experimental, predicted)
# dmat.to_csv("distance_matrix.csv")
# dmat.head()

## 5. Binary similarity matrix and candidate scoring
Distances below a threshold count as a "hit". Each candidate is scored by how many
experimental genes it is similar to; higher scores are stronger candidates.

In [ ]:
def prioritize(dmat, threshold_frac=0.01, min_frequency=6):
    """Threshold the distance matrix and rank candidates by number of hits."""
    threshold = threshold_frac * (dmat.to_numpy().max() - dmat.to_numpy().min())
    binary = (dmat > threshold).astype(int)          # 0 = similar (hit), 1 = far
    frequency = (1 - binary).sum(axis=0)             # count of hits per candidate
    ranked = frequency[frequency >= min_frequency].sort_values(ascending=False)
    return ranked.rename("hits").to_frame()


# Example:
# ranking = prioritize(dmat)
# ranking.head(14)   # top candidate genes

## 6. Dendrogram
Hierarchical clustering of the prioritized candidates for visual inspection.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

def plot_dendrogram(dmat, method="single"):
    linkage_matrix = linkage(dmat.values, method)
    plt.figure(figsize=(15, 8))
    dendrogram(linkage_matrix, labels=list(dmat.index))
    plt.title("Candidate gene dendrogram")
    plt.tight_layout()
    plt.show()

# Example:
# plot_dendrogram(dmat)